Generate datasets for emotion concepts

In [ ]:
import json
import sys
import time
from dataclasses import asdict
from pathlib import Path
from typing import Iterable, Iterator, Sequence

import torch
import yaml

# Must come BEFORE the core.* imports below — this notebook lives two levels
# down, so `core` is only importable once src/ is on the path. Anchored to the
# project root, not the kernel's cwd, which Jupyter front-ends disagree about.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.corpus import (
    build_instruction,
    build_prompts,
    generate_corpus,
    generate_stories,
)
from core.shards import read_jsonl, read_shards, write_jsonl, write_metadata
from core.types import Prompt, Story
from core.models import Model
from core.utils import mentions_emotion

In [ ]:
MODEL_PATH = "Qwen/Qwen2.5-0.5B-Instruct"

CACHE_DIR = PROJECT_ROOT / ".cache"
DATA_DIR = PROJECT_ROOT / "datasets" / "qwen-emotion-stories"
SEED = 0

model = Model(MODEL_PATH)
model.load_weights()
model.device

In [ ]:
CONFIG_PATH = DATA_DIR / "config.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

EMOTIONS: tuple[str, ...] = tuple(CONFIG["emotions"])
TOPICS: tuple[str, ...] = tuple(CONFIG["topics"])
INTENSITIES: tuple[dict, ...] = tuple(CONFIG["intensity"])
IMPLICIT: tuple[dict, ...] = tuple(CONFIG["implicit"])

print(f"{len(EMOTIONS)} emotions, {len(TOPICS)} topics, "
      f"{len(INTENSITIES)} ladders, {len(IMPLICIT)} implicit scenarios")
print(f"corpus: {len(EMOTIONS) + 1} shards x {len(TOPICS)} topics x n_per_pair")

In [ ]:
# inspect
prompts = build_prompts(EMOTIONS, TOPICS, n_per_pair=1)
print(f"{len(prompts)} prompts\n")
for p in prompts[:6]:
    print(f"[{p.emotion} / {p.topic} / {p.index}]\n{p.instruction}\n")

In [ ]:
resp = model.gen(prompts[:2], sample=True)

In [ ]:
resp[0]

In [ ]:
for s in generate_stories(model, prompts[:5]):
    print(s.text)

## Shards

IO lives in `core/shards.py`. One file per emotion, written atomically, so a crashed run costs one emotion and the resume check is just "does this file exist".

## Run

`limit` writes `<emotion>.sample` shards, which `read_shards` filters out — so the eyeball pass cannot contaminate the corpus.

In [ ]:
# eyeball pass — 2 stories per emotion, written to *.sample shards
# (read_shards filters those out, so it cannot contaminate the corpus)
generate_corpus(model, DATA_DIR, emotions=EMOTIONS, topics=TOPICS, limit=2)

In [ ]:
# the real run — 1,300 stories. Prefer the terminal for this:
#   caffeinate -i uv run python src/scripts/generate_corpus.py
counts = generate_corpus(model, DATA_DIR, emotions=EMOTIONS, topics=TOPICS, n_per_pair=4)
write_metadata(DATA_DIR, model=MODEL_PATH, seed=SEED, batch_size=16,
               n_per_pair=4, emotions=list(EMOTIONS), topics=list(TOPICS), counts=counts)
counts

## Publish

The notebook's job ends with the shards. `package_dataset.py` assigns the split,
derives the held-out sets from `config.yaml`, and renders `README.md`; `hf upload`
syncs the folder. The folder is the Hub repo — what you see locally is published.

```bash
uv run python src/scripts/package_dataset.py
hf upload foogunlana/qwen-emotion-stories datasets/qwen-emotion-stories \
    --repo-type=dataset --delete "*"
```